In [1]:
import numpy as np
from scipy.optimize import minimize

## Solution 1(a) 

In [3]:
# Abstract Problem class

class Problem:
    def value(self, x):
        raise Exception("Not implemented!")

    def gradient(self, x):
        raise Exception("Not implemented!")

    def hessian(self, x):
        raise Exception("Not implemented!")

In [4]:
# Helper function for matrix T_n

def Tn(n):
    Q = 2 * np.eye(n)

    for i in range(n - 1):
        Q[i, i + 1] = -1
        Q[i + 1, i] = -1

    return Q

In [5]:
# Function f1: Himmelblau function

class Himmelblau(Problem):

    def value(self, x):
        x1, x2 = x
        return (x1**2 + x2 - 11)**2 + (x1 + x2**2 - 7)**2

    def gradient(self, x):
        x1, x2 = x

        df_dx1 = 4*x1*(x1**2 + x2 - 11) + 2*(x1 + x2**2 - 7)
        df_dx2 = 2*(x1**2 + x2 - 11) + 4*x2*(x1 + x2**2 - 7)

        return np.array([df_dx1, df_dx2])

    def hessian(self, x):
        x1, x2 = x

        return np.array([
            [12*x1**2 + 4*x2 - 42, 4*x1 + 4*x2],
            [4*x1 + 4*x2, 4*x1 + 12*x2**2 - 26]
        ])

In [6]:
# Function f2: Quadratic function

class QuadraticFunction(Problem):

    def __init__(self, n):
        self.Q = Tn(n)
        self.p = np.ones(n)
        self.c = 0

    def value(self, x):
        return 0.5 * x @ self.Q @ x + self.p @ x + self.c

    def gradient(self, x):
        return self.Q @ x + self.p

    def hessian(self, x):
        return self.Q

In [7]:
# Function f3

class Function3(Problem):

    def __init__(self, n):
        self.Q = Tn(n)

    def value(self, x):
        return np.sqrt(1 + x @ self.Q @ x)

    def gradient(self, x):
        qx = self.Q @ x
        return qx / np.sqrt(1 + x @ qx)

    def hessian(self, x):
        qx = self.Q @ x
        s = 1 + x @ qx

        return self.Q / np.sqrt(s) - np.outer(qx, qx) / s**1.5

In [8]:
# Function f4

class Function4(Problem):

    def __init__(self, n):
        self.Q = Tn(n)

    def value(self, x):
        q = x @ self.Q @ x
        return np.sqrt(1 + q) + 0.5*q

    def gradient(self, x):
        qx = self.Q @ x
        return qx / np.sqrt(1 + x @ qx) + qx

    def hessian(self, x):
        qx = self.Q @ x
        s = 1 + x @ qx

        return (
            self.Q / np.sqrt(s)
            - np.outer(qx, qx) / s**1.5
            + self.Q
        )

In [9]:
# Testing the functions

f1 = Himmelblau()
f2 = QuadraticFunction(3)
f3 = Function3(3)
f4 = Function4(3)

x2 = np.array([1.0, 2.0])
x3 = np.array([1.0, 1.0, 1.0])

print("f1 =", f1.value(x2))
print("grad f1 =", f1.gradient(x2))

print("f2 =", f2.value(x3))
print("grad f2 =", f2.gradient(x3))

print("f3 =", f3.value(x3))
print("grad f3 =", f3.gradient(x3))

print("f4 =", f4.value(x3))
print("grad f4 =", f4.gradient(x3))

f1 = 68.0
grad f1 = [-36. -32.]
f2 = 4.0
grad f2 = [2. 1. 2.]
f3 = 1.7320508075688772
grad f3 = [0.57735027 0.         0.57735027]
f4 = 2.732050807568877
grad f4 = [1.57735027 0.         1.57735027]


## Solution 2.1 

In [10]:
# Minimize Himmelblau function with different starting values

f1 = Himmelblau()

start_values = [
    [0, 0],
    [4, 4],
    [-4, 4],
    [-4, -4],
    [4, -4]
]

for x0 in start_values:

    result = minimize(
        f1.value,
        x0,
        jac=f1.gradient,
        method="BFGS"
    )

    print("Start =", x0)
    print("Minimum =", result.x)
    print("Function value =", result.fun)
    print()

Start = [0, 0]
Minimum = [2.99999995 2.        ]
Function value = 1.0569835706588071e-13

Start = [4, 4]
Minimum = [3.00000002 2.00000001]
Function value = 1.425092090918083e-14

Start = [-4, 4]
Minimum = [-2.80511802  3.13131252]
Function value = 1.2948648805565608e-13

Start = [-4, -4]
Minimum = [-3.77931026 -3.28318599]
Function value = 3.296541485744662e-15

Start = [4, -4]
Minimum = [ 3.58442835 -1.84812653]
Function value = 5.355954901698142e-15



In [ ]:
# Solution 2.1 - Observation
# Different starting values can converge to different minima.
# Himmelblau's function has four minima.

## Solution 2.2

In [11]:
# Compare CG, Newton-CG and BFGS for f3

n = 5
f3 = Function3(n)

x0 = np.ones(n)

methods = ["CG", "Newton-CG", "BFGS"]
tolerances = [1e-3, 1e-6, 1e-9]

for tol in tolerances:

    print("Tolerance =", tol)

    for method in methods:

        if method == "Newton-CG":
            result = minimize(
                f3.value,
                x0,
                jac=f3.gradient,
                hess=f3.hessian,
                method=method,
                tol=tol
            )

        else:
            result = minimize(
                f3.value,
                x0,
                jac=f3.gradient,
                method=method,
                tol=tol
            )

        print(method)
        print("x =", result.x)
        print("f(x) =", result.fun)
        print("iterations =", result.nit)
        print()

    print("----------------------")

Tolerance = 0.001
CG
x = [0.00011524 0.00018251 0.00020792 0.00018251 0.00011524]
f(x) = 1.0000000184508036
iterations = 11

Newton-CG
x = [1.58879993e-10 3.25737832e-10 4.92595678e-10 3.25737839e-10
 1.58879993e-10]
f(x) = 1.0
iterations = 4

BFGS
x = [-3.35150681e-05 -1.16065004e-04 -4.56425447e-04 -1.16065004e-04
 -3.35150681e-05]
f(x) = 1.0000001237829754
iterations = 7

----------------------
Tolerance = 1e-06
CG
x = [6.43827147e-07 5.76812336e-07 7.94843839e-07 5.76812336e-07
 6.43827147e-07]
f(x) = 1.0000000000004665
iterations = 14

Newton-CG
x = [1.58879993e-10 3.25737832e-10 4.92595678e-10 3.25737839e-10
 1.58879993e-10]
f(x) = 1.0
iterations = 5

BFGS
x = [-1.97260696e-07  6.78309580e-08 -5.79332017e-08  6.78309585e-08
 -1.97260697e-07]
f(x) = 1.000000000000125
iterations = 10

----------------------
Tolerance = 1e-09
CG
x = [-1.16193656e-10 -7.33635554e-10 -2.17068319e-09 -7.33635557e-10
 -1.16193659e-10]
f(x) = 1.0
iterations = 21

Newton-CG
x = [1.58879993e-10 3.25737832e

In [12]:
# Compare CG, Newton-CG and BFGS for f4

n = 5
f4 = Function4(n)

x0 = np.ones(n)

methods = ["CG", "Newton-CG", "BFGS"]
tolerances = [1e-3, 1e-6, 1e-9]

for tol in tolerances:

    print("Tolerance =", tol)

    for method in methods:

        if method == "Newton-CG":
            result = minimize(
                f4.value,
                x0,
                jac=f4.gradient,
                hess=f4.hessian,
                method=method,
                tol=tol
            )

        else:
            result = minimize(
                f4.value,
                x0,
                jac=f4.gradient,
                method=method,
                tol=tol
            )

        print(method)
        print("x =", result.x)
        print("f(x) =", result.fun)
        print("iterations =", result.nit)
        print()

    print("----------------------")

Tolerance = 0.001
CG
x = [-0.00032283 -0.00068867 -0.00090903 -0.00068867 -0.00032283]
f(x) = 1.0000005732372197
iterations = 9

Newton-CG
x = [-6.32342149e-09 -7.56377594e-09 -8.80413038e-09 -7.56377594e-09
 -6.32342149e-09]
f(x) = 1.0
iterations = 5

BFGS
x = [ 3.80989455e-05 -1.90195881e-04 -1.03226855e-04 -1.90195881e-04
  3.80989455e-05]
f(x) = 1.0000001222673363
iterations = 7

----------------------
Tolerance = 1e-06
CG
x = [-2.54111183e-08 -1.88047141e-08 -7.08423415e-08 -1.88047141e-08
 -2.54111183e-08]
f(x) = 1.0000000000000067
iterations = 18

Newton-CG
x = [-6.32342149e-09 -7.56377594e-09 -8.80413038e-09 -7.56377594e-09
 -6.32342149e-09]
f(x) = 1.0
iterations = 5

BFGS
x = [-3.45930811e-09 -3.35817150e-08 -5.03903357e-08 -3.35821131e-08
 -3.45891028e-09]
f(x) = 1.0000000000000022
iterations = 10

----------------------
Tolerance = 1e-09
CG
x = [-7.96217158e-10 -2.35707286e-09 -6.74854460e-09 -2.35707286e-09
 -7.96217158e-10]
f(x) = 1.0
iterations = 22

Newton-CG
x = [-3.781

### Observation

Newton-CG converges much faster than CG and BFGS, reaching the exact solution (f(x) = 1.0) in only 4–7 iterations regardless of the tolerance. Tightening the tolerance from 1e-3 to 1e-9 barely changes its iteration count, since it uses the exact Hessian and therefore has quadratic convergence near the minimum.

CG and BFGS, which rely only on gradient information, need more iterations as the tolerance gets tighter (e.g. CG goes from 11 to 14 to 21 iterations as tol decreases from 1e-3 to 1e-9 for f3). Their convergence is slower than Newton-CG's, so higher precision costs noticeably more work, while Newton-CG's cost stays roughly constant.